# Figure 1: measurements and fits

This notebook makes **Figure 1** of the paper from the released data product
[`data/example_measurements.csv`](../data/README.md).

| | |
|---|---|
| **Input** | `data/example_measurements.csv` |
| **Output** | `paper/figure_1_measurements.pdf` (and `.png`) |
| **Used in paper** | `\includegraphics{figure_1_measurements.pdf}` |

The output is written straight into `paper/`, so re-running this notebook
and syncing to Overleaf (see [](../guide/overleaf.md)) updates the figure in
the paper with no manual copying.

In [ ]:
import sys
from pathlib import Path

# Make the repository's shared helpers importable from this notebook
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from common import paths
from common.text import count_macros

paths.use_paper_style()
data = np.genfromtxt(paths.DATA_DIR / "example_measurements.csv",
                     delimiter=",", names=True)
group_names = "ABCD"
groups = np.unique(data["group"]).astype(int)

## Fit each group

A weighted straight-line fit to the measurements of each group.

In [ ]:
def fit_line(x, y, y_err):
    """Weighted least-squares straight-line fit.

    Returns (slope, intercept), their 1-sigma uncertainties and the
    chi-squared per degree of freedom.
    """
    coeffs, cov = np.polyfit(x, y, 1, w=1 / y_err, cov="unscaled")
    resid = (y - np.polyval(coeffs, x)) / y_err
    chi2_dof = np.sum(resid**2) / (x.size - 2)
    return coeffs, np.sqrt(np.diag(cov)), chi2_dof

fits = {}
for g in groups:
    sel = data["group"] == g
    fits[g] = fit_line(data["x"][sel], data["y"][sel], data["y_err"][sel])

## Make the figure

In [ ]:
fig, ax = plt.subplots()
x_fine = np.linspace(data["x"].min(), data["x"].max(), 100)
for g in groups:
    sel = data["group"] == g
    line = ax.errorbar(data["x"][sel], data["y"][sel], yerr=data["y_err"][sel],
                       fmt=".", label=f"Group {group_names[g]}")
    (coeffs, _, _) = fits[g]
    ax.plot(x_fine, np.polyval(coeffs, x_fine), color=line[0].get_color())

ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.legend(loc="upper left", ncols=2)

out = paths.save_paper_figure(fig, "figure_1_measurements")
print(f"Saved {out.relative_to(paths.REPO_ROOT)}")